# FE-CLIP training on Kaggle (paper-faithful reconstruction)

This notebook reconstructs **FE-CLIP: Frequency Enhanced CLIP Model for Zero-Shot Anomaly Detection and Segmentation** (ICCV 2025) from the [official paper](https://openaccess.thecvf.com/content/ICCV2025/papers/Gong_FE-CLIP_Frequency_Enhanced_CLIP_Model_for_Zero-Shot_Anomaly_Detection_and_ICCV_2025_paper.pdf) and [supplement](https://openaccess.thecvf.com/content/ICCV2025/supplemental/Gong_FE-CLIP_Frequency_Enhanced_ICCV_2025_supplemental.pdf). The authors have not released code or checkpoints, so this is an independent implementation, not official author code.

Paper-specified settings used here: OpenAI CLIP ViT-L/14@336px; frozen CLIP image/text encoders; four visual stages; one FFE and one LFS adapter per stage; orthonormal DCT; `P=3`, `Q=3`, and `lambda=0.1`; prompts `A photo of a normal object` and `A photo of a damaged object`; image BCE plus pixel focal and Dice losses; Adam with learning rate `5e-4`; total batch size 16; and 9 epochs. Training follows the paper's cross-dataset protocol: train on MVTec AD test images for evaluation on VisA/other datasets, and train on VisA test images for evaluation on MVTec AD.

The authors report PyTorch 1.13.0 on four RTX 3090 24-GB GPUs and average results over five runs. A current Kaggle runtime uses a newer PyTorch build; the two-T4 `torchrun`/DDP setup below preserves the reported total batch size and FP32 remains the default. The five seeds were not published.

The paper does **not** specify the four exact layer endpoints, LFS convolution kernel, DCT normalization, optimizer betas, focal-loss parameters, image-resize rule, seed values, AMP use, parameter initialization, or whether CLIP LayerNorm is part of the intermediate class-token projection. It also calls the patch-alignment `fc` learnable in Section 3.2 while Sections 3.5/4.2 say only FFE/LFS parameters are optimized. This notebook trains that `fc`, because leaving a randomly initialized alignment layer frozen would make the segmentation objective ill-defined. All such choices are centralized below and recorded in every checkpoint. Defaults use the natural paper/AnomalyCLIP-compatible reconstruction: layers `[6, 12, 18, 24]`, orthonormal DCT, a 1x1 LFS channel-mixing convolution, Adam defaults, AnomalyCLIP's focal/Dice definitions, direct 336x336 resize, CLIP's frozen LayerNorm+projection for class tokens, FP32, and seed 111. Reproducing the authors' exact unpublished numbers cannot be guaranteed without their source code.

In [ ]:
from pathlib import Path

WORKING_DIR = Path('/kaggle/working')
MVTEC_PATH = Path('/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection')
VISA_PATH = Path('/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922')
OUTPUT_ROOT = WORKING_DIR / 'feclip_checkpoints'

# Paper protocol: MVTec-trained weights evaluate VisA and other datasets;
# VisA-trained weights evaluate MVTec. Run this notebook once per source dataset.
TRAIN_DATASET = 'MVTec'  # 'MVTec' or 'VisA'

# Values explicitly reported by the paper/supplement.
IMAGE_SIZE = 336
EPOCHS = 9
LEARNING_RATE = 5e-4
TOTAL_BATCH_SIZE = 16
FFE_WINDOW = 3          # P
LFS_WINDOW = 3          # Q
FREQUENCY_LAMBDA = 0.1
NUM_STAGES = 4
NORMAL_PROMPT = 'A photo of a normal object'
ABNORMAL_PROMPT = 'A photo of a damaged object'

# Kaggle memory/distributed controls. Select the T4 x2 accelerator for the default.
# Accelerate/DDP gives each T4 a micro-batch of 1 and accumulates eight times, so
# 2 GPUs x 1 sample x 8 accumulation steps = the paper's total batch size of 16.
NUM_TRAINING_PROCESSES = 2  # set to 1 for a P100/single-GPU runtime
GPU_MICRO_BATCH_SIZE = 1
NUM_WORKERS = 2
USE_AMP = False  # the paper does not report mixed precision; False is faithful
SEEDS = (111,)   # paper averages 5 runs but does not publish the five seeds

# Unpublished implementation choices, kept explicit and checkpointed.
STAGE_ENDPOINTS = (6, 12, 18, 24)
DCT_NORM = 'ortho'
LFS_CONV_KERNEL = 1
FOCAL_GAMMA = 2.0
FOCAL_SMOOTH = 1e-5
DICE_SMOOTH = 1.0
ADAM_BETAS = (0.9, 0.999)  # torch.optim.Adam defaults; paper only says Adam
ADAM_EPS = 1e-8

OPENAI_CLIP_URL = 'https://openaipublic.azureedge.net/clip/models/3035c92b350959924f9f00213499208652fc7ea050643e8b385c2dac08641f02/ViT-L-14-336px.pt'
OPENAI_CLIP_SHA256 = '3035c92b350959924f9f00213499208652fc7ea050643e8b385c2dac08641f02'
CLIP_COMMIT = 'dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1'
CLIP_WEIGHT_PATH = WORKING_DIR / 'ViT-L-14-336px.pt'

assert IMAGE_SIZE // 14 == 24 and (IMAGE_SIZE // 14) % FFE_WINDOW == 0
assert len(STAGE_ENDPOINTS) == NUM_STAGES and STAGE_ENDPOINTS[-1] == 24
assert TOTAL_BATCH_SIZE % (NUM_TRAINING_PROCESSES * GPU_MICRO_BATCH_SIZE) == 0
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Training source:', TRAIN_DATASET)
print('Checkpoint is for target:', 'MVTec' if TRAIN_DATASET == 'VisA' else 'VisA and other datasets')
print('Output:', OUTPUT_ROOT)

In [ ]:
# Kaggle internet must be enabled for the official OpenAI CLIP package/weight.
%pip install -q "accelerate==1.12.0" "ftfy==6.2.3" "regex==2024.11.6" "tqdm==4.67.1" "git+https://github.com/openai/CLIP.git@dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1"

In [ ]:
import csv
import hashlib
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
import urllib.request
from dataclasses import asdict, dataclass
from typing import Optional

import clip
from accelerate import Accelerator
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF
from tqdm.auto import tqdm

CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)
IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.webp', '.JPG'}

def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def download_verified(url, target, expected_sha256):
    target = Path(target)
    if target.exists() and sha256(target) == expected_sha256:
        print('Verified existing weight:', target)
        return target
    target.parent.mkdir(parents=True, exist_ok=True)
    print('Downloading official OpenAI CLIP weight...')
    urllib.request.urlretrieve(url, target)
    actual = sha256(target)
    if actual != expected_sha256:
        target.unlink(missing_ok=True)
        raise RuntimeError(f'CLIP SHA256 mismatch: expected {expected_sha256}, got {actual}')
    return target

download_verified(OPENAI_CLIP_URL, CLIP_WEIGHT_PATH, OPENAI_CLIP_SHA256)
print('Torch:', torch.__version__)
print('Requested training processes:', NUM_TRAINING_PROCESSES)
print('CUDA is initialized only inside the standalone torchrun workers.')

## Cross-dataset training data

The paper deliberately trains on the labeled **test split** of an auxiliary anomaly dataset. This requires both image labels and pixel masks. Normal samples receive an all-zero mask. MVTec is read from its native `category/test/...` layout. VisA is read from the official `split_csv/1cls.csv`, preserving its official test split.

In [ ]:
@dataclass(frozen=True)
class Sample:
    image_path: Path
    mask_path: Optional[Path]
    label: int
    category: str

def image_files(folder):
    return sorted(path for path in folder.rglob('*') if path.is_file() and path.suffix in IMAGE_EXTENSIONS)

def mvtec_test_samples(root):
    root = Path(root)
    samples = []
    for category_dir in sorted(path for path in root.iterdir() if (path / 'test').is_dir()):
        for defect_dir in sorted(path for path in (category_dir / 'test').iterdir() if path.is_dir()):
            anomalous = defect_dir.name.lower() != 'good'
            for image_path in image_files(defect_dir):
                mask_path = None
                if anomalous:
                    relative = image_path.relative_to(defect_dir)
                    candidates = [
                        category_dir / 'ground_truth' / defect_dir.name / relative.with_name(relative.stem + '_mask.png'),
                        category_dir / 'ground_truth' / defect_dir.name / relative.with_suffix('.png'),
                    ]
                    mask_path = next((path for path in candidates if path.exists()), None)
                    if mask_path is None:
                        raise FileNotFoundError(f'Missing MVTec anomaly mask for {image_path}')
                samples.append(Sample(image_path, mask_path, int(anomalous), category_dir.name))
    if not samples:
        raise RuntimeError(f'No MVTec test samples found under {root}')
    return samples

def resolve_visa_path(root, value):
    value = str(value or '').strip()
    if not value:
        return None
    raw = Path(value)
    candidates = [raw] if raw.is_absolute() else [root / raw]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None

def visa_test_samples(root):
    root = Path(root)
    split_file = root / 'split_csv' / '1cls.csv'
    if not split_file.exists():
        raise FileNotFoundError(f'Official VisA split not found: {split_file}')
    samples = []
    with split_file.open('r', encoding='utf-8-sig', newline='') as handle:
        for row in csv.DictReader(handle):
            if str(row.get('split', '')).strip().lower() != 'test':
                continue
            category = str(row.get('object', '')).strip()
            image_path = resolve_visa_path(root, row.get('image'))
            if image_path is None:
                raise FileNotFoundError(f"VisA image listed in split CSV is missing: {row.get('image')}")
            label_text = str(row.get('label', '')).strip().lower()
            anomalous = label_text not in {'normal', 'good', '0', 'false'}
            mask_path = resolve_visa_path(root, row.get('mask')) if anomalous else None
            if anomalous and mask_path is None:
                raise FileNotFoundError(f"VisA anomaly mask is missing: {row.get('mask')}")
            samples.append(Sample(image_path, mask_path, int(anomalous), category))
    if not samples:
        raise RuntimeError(f'No VisA test samples found through {split_file}')
    return samples

class FECLIPTrainingDataset(Dataset):
    def __init__(self, dataset_name, root, image_size=336):
        self.dataset_name = dataset_name
        self.root = Path(root)
        self.image_size = image_size
        if not self.root.exists():
            raise FileNotFoundError(f'Dataset root does not exist: {self.root}')
        self.samples = mvtec_test_samples(self.root) if dataset_name == 'MVTec' else visa_test_samples(self.root)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        image = Image.open(sample.image_path).convert('RGB')
        image = TF.resize(image, [self.image_size, self.image_size], interpolation=InterpolationMode.BICUBIC, antialias=True)
        image = TF.to_tensor(image)
        image = TF.normalize(image, CLIP_MEAN, CLIP_STD)
        if sample.mask_path is None:
            mask = torch.zeros(1, self.image_size, self.image_size, dtype=torch.float32)
        else:
            mask_image = Image.open(sample.mask_path).convert('L')
            mask_image = TF.resize(mask_image, [self.image_size, self.image_size], interpolation=InterpolationMode.NEAREST)
            mask = (TF.pil_to_tensor(mask_image).float() > 0).float()
        return {'image': image, 'mask': mask, 'label': torch.tensor(sample.label), 'category': sample.category}


## FE-CLIP modules

FFE splits each 24x24 CLIP patch grid into non-overlapping 3x3 windows, applies DCT, a learned channel-wise linear layer plus GELU, and inverse DCT. LFS performs dense sliding-window 3x3 DCT, averages all nine frequency-response groups as required by the supplement, then applies a learned convolution plus GELU. At each stage, `f_hat = lambda * (FFE(f) + LFS(f)) + (1-lambda) * f`.

In [ ]:
def orthonormal_dct_matrix(size, dtype=torch.float32):
    matrix = torch.empty(size, size, dtype=dtype)
    scale0 = math.sqrt(1.0 / size)
    scale = math.sqrt(2.0 / size)
    for frequency in range(size):
        alpha = scale0 if frequency == 0 else scale
        for position in range(size):
            matrix[frequency, position] = alpha * math.cos(math.pi * (2 * position + 1) * frequency / (2 * size))
    return matrix

class FFEAdapter(nn.Module):
    def __init__(self, channels, window_size=3):
        super().__init__()
        self.window_size = window_size
        self.register_buffer('dct', orthonormal_dct_matrix(window_size), persistent=True)
        self.linear = nn.Linear(channels, channels)
        self.activation = nn.GELU()

    def forward(self, features):
        batch, channels, height, width = features.shape
        p = self.window_size
        if height % p or width % p:
            raise ValueError(f'FFE requires a patch grid divisible by P={p}, got {height}x{width}')
        blocks = features.unfold(2, p, p).unfold(3, p, p)  # B,C,H/P,W/P,P,P
        dct = self.dct.to(device=features.device, dtype=features.dtype)
        frequency = torch.einsum('ip,bcxypq,jq->bcxyij', dct, blocks, dct)
        frequency = frequency.permute(0, 2, 3, 4, 5, 1)
        frequency = self.activation(self.linear(frequency))
        frequency = frequency.permute(0, 5, 1, 2, 3, 4)
        spatial = torch.einsum('ip,bcxyij,jq->bcxypq', dct, frequency, dct)
        return spatial.permute(0, 1, 2, 4, 3, 5).reshape(batch, channels, height, width)

class LFSAdapter(nn.Module):
    def __init__(self, channels, window_size=3, conv_kernel=1):
        super().__init__()
        if window_size % 2 != 1:
            raise ValueError('Paper-faithful same-layout sliding DCT requires an odd Q')
        self.window_size = window_size
        self.register_buffer('dct', orthonormal_dct_matrix(window_size), persistent=True)
        self.conv = nn.Conv2d(channels, channels, conv_kernel, padding=conv_kernel // 2)
        self.activation = nn.GELU()

    def forward(self, features):
        batch, channels, height, width = features.shape
        q = self.window_size
        patches = F.unfold(features, kernel_size=q, padding=q // 2, stride=1)
        patches = patches.view(batch, channels, q, q, height, width)
        dct = self.dct.to(device=features.device, dtype=features.dtype)
        frequency = torch.einsum('ip,bcpqhw,jq->bcijhw', dct, patches, dct)
        mean_frequency = frequency.mean(dim=(2, 3))  # mean all QxQ response groups
        return self.activation(self.conv(mean_frequency))

class FocalLoss(nn.Module):
    # Matches the AnomalyCLIP loss used by the training protocol FE-CLIP follows.
    def __init__(self, gamma=2.0, smooth=1e-5):
        super().__init__()
        self.gamma = gamma
        self.smooth = smooth

    def forward(self, probabilities, target):
        classes = probabilities.shape[1]
        flattened = probabilities.permute(0, 2, 3, 1).reshape(-1, classes)
        target = target.squeeze(1).reshape(-1).long()
        one_hot = F.one_hot(target, num_classes=classes).to(flattened.dtype)
        one_hot = one_hot.clamp(self.smooth / (classes - 1), 1.0 - self.smooth)
        pt = (one_hot * flattened).sum(dim=1) + self.smooth
        return (-(1.0 - pt).pow(self.gamma) * pt.log()).mean()

class BinaryDiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, probability, target):
        probability = probability.reshape(probability.shape[0], -1)
        target = target.reshape(target.shape[0], -1)
        intersection = (probability * target).sum(dim=1)
        dice = (2.0 * intersection + self.smooth) / (probability.sum(dim=1) + target.sum(dim=1) + self.smooth)
        return 1.0 - dice.mean()


In [ ]:
class FECLIP(nn.Module):
    def __init__(self, clip_model, stage_endpoints=(6, 12, 18, 24), p=3, q=3, frequency_lambda=0.1, lfs_kernel=1):
        super().__init__()
        self.clip = clip_model
        self.stage_endpoints = tuple(stage_endpoints)
        self.frequency_lambda = frequency_lambda
        visual = self.clip.visual
        self.width = visual.conv1.out_channels
        self.embed_dim = visual.proj.shape[1]
        self.grid_size = visual.input_resolution // visual.conv1.kernel_size[0]
        self.ffe_adapters = nn.ModuleList([FFEAdapter(self.width, p) for _ in self.stage_endpoints])
        self.lfs_adapters = nn.ModuleList([LFSAdapter(self.width, q, lfs_kernel) for _ in self.stage_endpoints])
        # Section 3.2 calls this a single learnable fc shared across stage maps.
        self.patch_projection = nn.Linear(self.width, self.embed_dim, bias=True)
        for parameter in self.clip.parameters():
            parameter.requires_grad_(False)
        tokenized = clip.tokenize([NORMAL_PROMPT, ABNORMAL_PROMPT])
        with torch.no_grad():
            text = self.clip.encode_text(tokenized.to(next(self.clip.parameters()).device)).float()
            text = F.normalize(text, dim=-1)
        self.register_buffer('text_features', text, persistent=True)

    def trainable_parameters(self):
        modules = [self.ffe_adapters, self.lfs_adapters, self.patch_projection]
        return [parameter for module in modules for parameter in module.parameters()]

    def train(self, mode=True):
        super().train(mode)
        self.clip.eval()  # CLIP stays frozen/eval exactly as stated by the paper
        return self

    def _probabilities(self, visual_features):
        visual_features = F.normalize(visual_features.float(), dim=-1)
        scale = self.clip.logit_scale.exp().detach().float()
        return (scale * visual_features @ self.text_features.float().T).softmax(dim=-1)

    def forward(self, images):
        visual = self.clip.visual
        x = visual.conv1(images.to(dtype=visual.conv1.weight.dtype))
        batch, channels, grid_h, grid_w = x.shape
        if (grid_h, grid_w) != (self.grid_size, self.grid_size):
            raise ValueError(f'Expected {self.grid_size}x{self.grid_size} CLIP patch grid, got {grid_h}x{grid_w}')
        x = x.reshape(batch, channels, grid_h * grid_w).permute(0, 2, 1)
        class_token = visual.class_embedding.to(x.dtype) + torch.zeros(batch, 1, channels, device=x.device, dtype=x.dtype)
        x = torch.cat([class_token, x], dim=1)
        x = visual.ln_pre(x + visual.positional_embedding.to(x.dtype))
        x = x.permute(1, 0, 2)  # L,B,C for OpenAI CLIP residual blocks

        class_probabilities = []
        segmentation_probabilities = []
        block_start = 0
        for stage_index, block_end in enumerate(self.stage_endpoints):
            for block in visual.transformer.resblocks[block_start:block_end]:
                x = block(x)
            block_start = block_end
            cls = x[0]
            patches = x[1:].permute(1, 2, 0).reshape(batch, channels, grid_h, grid_w)
            frequency = self.ffe_adapters[stage_index](patches) + self.lfs_adapters[stage_index](patches)
            enhanced = self.frequency_lambda * frequency + (1.0 - self.frequency_lambda) * patches

            # Frozen standard CLIP class-token projection head at every stage.
            projected_cls = visual.ln_post(cls) @ visual.proj
            class_probabilities.append(self._probabilities(projected_cls))
            projected_patches = self.patch_projection(enhanced.permute(0, 2, 3, 1))
            segmentation_probabilities.append(self._probabilities(projected_patches))

            enhanced_tokens = enhanced.flatten(2).permute(2, 0, 1)
            x = torch.cat([cls.unsqueeze(0), enhanced_tokens], dim=0)

        return class_probabilities, segmentation_probabilities

def build_feclip(device):
    backbone, _ = clip.load(str(CLIP_WEIGHT_PATH), device='cpu', jit=False)
    backbone = backbone.float().to(device).eval()
    model = FECLIP(
        backbone, stage_endpoints=STAGE_ENDPOINTS, p=FFE_WINDOW, q=LFS_WINDOW,
        frequency_lambda=FREQUENCY_LAMBDA, lfs_kernel=LFS_CONV_KERNEL,
    ).to(device)
    model.train()
    return model


In [ ]:
# Structural checks before an expensive run.
with torch.no_grad():
    dct = orthonormal_dct_matrix(3)
    assert torch.allclose(dct @ dct.T, torch.eye(3), atol=1e-6)
    probe = torch.randn(2, 8, 24, 24)
    assert FFEAdapter(8, 3)(probe).shape == probe.shape
    assert LFSAdapter(8, 3, LFS_CONV_KERNEL)(probe).shape == probe.shape
print('DCT and adapter shape checks passed.')

print('Full frozen-backbone and trainable-parameter checks run inside each Accelerate worker.')

In [ ]:
dataset_root = MVTEC_PATH if TRAIN_DATASET == 'MVTec' else VISA_PATH
train_dataset = FECLIPTrainingDataset(TRAIN_DATASET, dataset_root, IMAGE_SIZE)
labels = np.array([sample.label for sample in train_dataset.samples])
categories = sorted({sample.category for sample in train_dataset.samples})
print('Samples:', len(train_dataset))
print('Normal/anomalous:', int((labels == 0).sum()), int((labels == 1).sum()))
print('Categories:', categories)
optimizer_steps_per_epoch = math.ceil(len(train_dataset) / TOTAL_BATCH_SIZE)
print('Approximate optimizer steps per epoch:', optimizer_steps_per_epoch)
print('Approximate optimizer steps for all 9 epochs:', optimizer_steps_per_epoch * EPOCHS)
first = train_dataset[0]
assert first['image'].shape == (3, IMAGE_SIZE, IMAGE_SIZE)
assert first['mask'].shape == (1, IMAGE_SIZE, IMAGE_SIZE)


## Nine-epoch training

For each stage, image loss is binary cross-entropy on the abnormal probability. Pixel loss is focal loss on the upsampled two-class map plus Dice loss on the abnormal channel. Both are averaged across four stages, matching equations (4) and (5), and `L_total = L_cls + L_mask`.

In [ ]:
focal_loss = FocalLoss(FOCAL_GAMMA, FOCAL_SMOOTH)
dice_loss = BinaryDiceLoss(DICE_SMOOTH)

def feclip_loss(class_probabilities, segmentation_probabilities, labels, masks):
    labels_float = labels.float()
    class_losses = [F.binary_cross_entropy(probs[:, 1].clamp(1e-6, 1 - 1e-6), labels_float) for probs in class_probabilities]
    mask_losses = []
    for probs in segmentation_probabilities:
        probs = probs.permute(0, 3, 1, 2)
        probs = F.interpolate(probs, size=masks.shape[-2:], mode='bilinear', align_corners=False)
        mask_losses.append(focal_loss(probs, masks) + dice_loss(probs[:, 1:2], masks))
    loss_cls = torch.stack(class_losses).mean()
    loss_mask = torch.stack(mask_losses).mean()
    return loss_cls + loss_mask, loss_cls.detach(), loss_mask.detach()

def checkpoint_config(seed):
    return {
        'paper': 'Gong et al., FE-CLIP, ICCV 2025',
        'implementation': 'independent_paper_reconstruction',
        'train_dataset': TRAIN_DATASET,
        'intended_target': 'MVTec' if TRAIN_DATASET == 'VisA' else 'VisA_and_other_datasets',
        'backbone': 'OpenAI CLIP ViT-L/14@336px',
        'clip_sha256': OPENAI_CLIP_SHA256,
        'image_size': IMAGE_SIZE, 'epochs': EPOCHS, 'learning_rate': LEARNING_RATE,
        'total_batch_size': TOTAL_BATCH_SIZE, 'gpu_micro_batch_size': GPU_MICRO_BATCH_SIZE,
        'optimizer': 'Adam', 'adam_betas': ADAM_BETAS, 'adam_eps': ADAM_EPS,
        'stage_endpoints': STAGE_ENDPOINTS, 'ffe_window_p': FFE_WINDOW, 'lfs_window_q': LFS_WINDOW,
        'frequency_lambda': FREQUENCY_LAMBDA, 'dct_norm': DCT_NORM, 'lfs_conv_kernel': LFS_CONV_KERNEL,
        'normal_prompt': NORMAL_PROMPT, 'abnormal_prompt': ABNORMAL_PROMPT,
        'focal_gamma': FOCAL_GAMMA, 'focal_smooth': FOCAL_SMOOTH, 'dice_smooth': DICE_SMOOTH,
        'use_amp': USE_AMP, 'seed': seed, 'torch_version': torch.__version__,
        'paper_underspecified_choices': [
            'stage_endpoints', 'dct_norm', 'lfs_conv_kernel', 'optimizer_betas_and_eps',
            'focal_parameters', 'resize_rule', 'seed', 'parameter_initialization',
            'intermediate_class_projection_layernorm', 'learnable_patch_fc_optimization',
        ],
    }

def train_one_seed(seed, accelerator, gradient_accumulation_steps):
    set_seed(seed)
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(
        train_dataset, batch_size=GPU_MICRO_BATCH_SIZE, shuffle=True, generator=generator,
        num_workers=NUM_WORKERS, pin_memory=True, drop_last=False,
    )
    model = build_feclip(accelerator.device)
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    frozen = sum(parameter.numel() for parameter in model.parameters() if not parameter.requires_grad)
    assert all(not parameter.requires_grad for parameter in model.clip.parameters())
    assert len(model.ffe_adapters) == len(model.lfs_adapters) == 4
    optimizer = torch.optim.Adam(
        model.trainable_parameters(), lr=LEARNING_RATE, betas=ADAM_BETAS, eps=ADAM_EPS, weight_decay=0.0,
    )
    model, optimizer, loader = accelerator.prepare(model, optimizer, loader)
    run_dir = OUTPUT_ROOT / f'train_on_{TRAIN_DATASET.lower()}_seed_{seed}'
    if accelerator.is_main_process:
        run_dir.mkdir(parents=True, exist_ok=True)
        print(f'DDP workers: {accelerator.num_processes}; accumulation: {gradient_accumulation_steps}')
        print(f'Trainable parameters: {trainable:,}; frozen CLIP parameters: {frozen:,}')
    accelerator.wait_for_everyone()
    history = []

    for epoch in range(1, EPOCHS + 1):
        epoch_started = time.perf_counter()
        model.train()
        if hasattr(loader, 'set_epoch'):
            loader.set_epoch(epoch)
        local_totals = torch.zeros(4, device=accelerator.device, dtype=torch.float64)
        progress = tqdm(
            loader, desc=f'seed {seed} epoch {epoch}/{EPOCHS}',
            disable=not accelerator.is_local_main_process,
        )
        optimizer.zero_grad(set_to_none=True)
        for batch in progress:
            with accelerator.accumulate(model):
                with accelerator.autocast():
                    class_probs, segmentation_probs = model(batch['image'])
                    loss, loss_cls, loss_mask = feclip_loss(
                        class_probs, segmentation_probs, batch['label'], batch['mask'],
                    )
                accelerator.backward(loss)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
            count = batch['image'].shape[0]
            local_totals += torch.stack([
                loss.detach() * count, loss_cls * count, loss_mask * count,
                torch.tensor(count, device=accelerator.device, dtype=loss.dtype),
            ]).to(torch.float64)
            if accelerator.sync_gradients:
                progress.set_postfix(loss=f"{loss.detach().item():.4f}")

        totals = accelerator.reduce(local_totals, reduction='sum')
        if accelerator.is_main_process:
            sample_count = int(totals[3].item())
            record = {
                'loss': totals[0].item() / sample_count,
                'cls': totals[1].item() / sample_count,
                'mask': totals[2].item() / sample_count,
                'samples': sample_count, 'epoch': epoch,
            }
            record['epoch_minutes'] = (time.perf_counter() - epoch_started) / 60.0
            record['estimated_remaining_hours'] = record['epoch_minutes'] * (EPOCHS - epoch) / 60.0
            history.append(record)
            core_model = accelerator.unwrap_model(model)
            config = checkpoint_config(seed)
            config.update({
                'distributed_backend': 'Accelerate_DDP',
                'num_training_processes': accelerator.num_processes,
                'gradient_accumulation_steps': gradient_accumulation_steps,
            })
            checkpoint = {
                'epoch': epoch, 'config': config, 'history': history,
                'ffe_adapters': core_model.ffe_adapters.state_dict(),
                'lfs_adapters': core_model.lfs_adapters.state_dict(),
                'patch_projection': core_model.patch_projection.state_dict(),
                'optimizer': optimizer.state_dict(),
            }
            checkpoint_path = run_dir / f'feclip_train_on_{TRAIN_DATASET.lower()}_epoch_{epoch:02d}.pth'
            torch.save(checkpoint, checkpoint_path)
            (run_dir / 'history.json').write_text(json.dumps(history, indent=2), encoding='utf-8')
            print(record)
        accelerator.wait_for_everyone()

    del model, optimizer, loader
    accelerator.free_memory()

def distributed_training_entrypoint():
    mixed_precision = 'fp16' if USE_AMP else 'no'
    accumulation = TOTAL_BATCH_SIZE // (NUM_TRAINING_PROCESSES * GPU_MICRO_BATCH_SIZE)
    accelerator = Accelerator(gradient_accumulation_steps=accumulation, mixed_precision=mixed_precision)
    if accelerator.num_processes != NUM_TRAINING_PROCESSES:
        raise RuntimeError(
            f'Requested {NUM_TRAINING_PROCESSES} processes but Accelerate started {accelerator.num_processes}. '
            'Select Kaggle T4 x2 or set NUM_TRAINING_PROCESSES=1.'
        )
    for seed in SEEDS:
        train_one_seed(seed, accelerator, accumulation)
    accelerator.end_training()

def load_feclip_checkpoint(checkpoint_path, device=torch.device('cpu')):
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model = build_feclip(device)
    model.ffe_adapters.load_state_dict(checkpoint['ffe_adapters'])
    model.lfs_adapters.load_state_dict(checkpoint['lfs_adapters'])
    model.patch_projection.load_state_dict(checkpoint['patch_projection'])
    return model.eval(), checkpoint


In [ ]:
# Write a real Python program and let torchrun start two clean DDP interpreters.
# All editable parent settings are passed through JSON; the worker has no stale dataset default.
training_script_source = ''.join([
    'from pathlib import Path\n',
    'import json\n',
    'import os\n',
    '\n',
    "runtime_config = json.loads(os.environ['FECLIP_RUNTIME_CONFIG_JSON'])\n",
    "WORKING_DIR = Path(runtime_config['WORKING_DIR'])\n",
    "MVTEC_PATH = Path(runtime_config['MVTEC_PATH'])\n",
    "VISA_PATH = Path(runtime_config['VISA_PATH'])\n",
    "OUTPUT_ROOT = Path(runtime_config['OUTPUT_ROOT'])\n",
    "TRAIN_DATASET = runtime_config['TRAIN_DATASET']\n",
    "IMAGE_SIZE = runtime_config['IMAGE_SIZE']\n",
    "EPOCHS = runtime_config['EPOCHS']\n",
    "LEARNING_RATE = runtime_config['LEARNING_RATE']\n",
    "TOTAL_BATCH_SIZE = runtime_config['TOTAL_BATCH_SIZE']\n",
    "FFE_WINDOW = runtime_config['FFE_WINDOW']\n",
    "LFS_WINDOW = runtime_config['LFS_WINDOW']\n",
    "FREQUENCY_LAMBDA = runtime_config['FREQUENCY_LAMBDA']\n",
    "NUM_STAGES = runtime_config['NUM_STAGES']\n",
    "NORMAL_PROMPT = runtime_config['NORMAL_PROMPT']\n",
    "ABNORMAL_PROMPT = runtime_config['ABNORMAL_PROMPT']\n",
    "NUM_TRAINING_PROCESSES = runtime_config['NUM_TRAINING_PROCESSES']\n",
    "GPU_MICRO_BATCH_SIZE = runtime_config['GPU_MICRO_BATCH_SIZE']\n",
    "NUM_WORKERS = runtime_config['NUM_WORKERS']\n",
    "USE_AMP = runtime_config['USE_AMP']\n",
    "SEEDS = tuple(runtime_config['SEEDS'])\n",
    "STAGE_ENDPOINTS = tuple(runtime_config['STAGE_ENDPOINTS'])\n",
    "DCT_NORM = runtime_config['DCT_NORM']\n",
    "LFS_CONV_KERNEL = runtime_config['LFS_CONV_KERNEL']\n",
    "FOCAL_GAMMA = runtime_config['FOCAL_GAMMA']\n",
    "FOCAL_SMOOTH = runtime_config['FOCAL_SMOOTH']\n",
    "DICE_SMOOTH = runtime_config['DICE_SMOOTH']\n",
    "ADAM_BETAS = tuple(runtime_config['ADAM_BETAS'])\n",
    "ADAM_EPS = runtime_config['ADAM_EPS']\n",
    "OPENAI_CLIP_URL = runtime_config['OPENAI_CLIP_URL']\n",
    "OPENAI_CLIP_SHA256 = runtime_config['OPENAI_CLIP_SHA256']\n",
    "CLIP_COMMIT = runtime_config['CLIP_COMMIT']\n",
    "CLIP_WEIGHT_PATH = Path(runtime_config['CLIP_WEIGHT_PATH'])\n",
    '\n',
    "assert TRAIN_DATASET in {'MVTec', 'VisA'}\n",
    'assert IMAGE_SIZE // 14 == 24 and (IMAGE_SIZE // 14) % FFE_WINDOW == 0\n',
    'assert len(STAGE_ENDPOINTS) == NUM_STAGES and STAGE_ENDPOINTS[-1] == 24\n',
    'assert TOTAL_BATCH_SIZE % (NUM_TRAINING_PROCESSES * GPU_MICRO_BATCH_SIZE) == 0\n',
    'OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)\n',
    "print('Training source:', TRAIN_DATASET)\n",
    "print('Checkpoint is for target:', 'MVTec' if TRAIN_DATASET == 'VisA' else 'VisA and other datasets')\n",
    "print('Output:', OUTPUT_ROOT)\n",
    '\n',
    'import csv\n',
    'import hashlib\n',
    'import json\n',
    'import math\n',
    'import os\n',
    'import random\n',
    'import shutil\n',
    'import subprocess\n',
    'import sys\n',
    'import time\n',
    'import urllib.request\n',
    'from dataclasses import asdict, dataclass\n',
    'from typing import Optional\n',
    '\n',
    'import clip\n',
    'from accelerate import Accelerator\n',
    'import numpy as np\n',
    'import torch\n',
    'import torch.nn as nn\n',
    'import torch.nn.functional as F\n',
    'from PIL import Image\n',
    'from torch.utils.data import DataLoader, Dataset\n',
    'from torchvision.transforms import InterpolationMode\n',
    'from torchvision.transforms import functional as TF\n',
    'from tqdm.auto import tqdm\n',
    '\n',
    'CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)\n',
    'CLIP_STD = (0.26862954, 0.26130258, 0.27577711)\n',
    "IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.webp', '.JPG'}\n",
    '\n',
    'def set_seed(seed):\n',
    "    os.environ['PYTHONHASHSEED'] = str(seed)\n",
    '    random.seed(seed)\n',
    '    np.random.seed(seed)\n',
    '    torch.manual_seed(seed)\n',
    '    torch.cuda.manual_seed_all(seed)\n',
    '    torch.backends.cudnn.deterministic = True\n',
    '    torch.backends.cudnn.benchmark = False\n',
    '\n',
    'def sha256(path):\n',
    '    digest = hashlib.sha256()\n',
    "    with open(path, 'rb') as handle:\n",
    "        for chunk in iter(lambda: handle.read(1024 * 1024), b''):\n",
    '            digest.update(chunk)\n',
    '    return digest.hexdigest()\n',
    '\n',
    'def download_verified(url, target, expected_sha256):\n',
    '    target = Path(target)\n',
    '    if target.exists() and sha256(target) == expected_sha256:\n',
    "        print('Verified existing weight:', target)\n",
    '        return target\n',
    '    target.parent.mkdir(parents=True, exist_ok=True)\n',
    "    print('Downloading official OpenAI CLIP weight...')\n",
    '    urllib.request.urlretrieve(url, target)\n',
    '    actual = sha256(target)\n',
    '    if actual != expected_sha256:\n',
    '        target.unlink(missing_ok=True)\n',
    "        raise RuntimeError(f'CLIP SHA256 mismatch: expected {expected_sha256}, got {actual}')\n",
    '    return target\n',
    '\n',
    'download_verified(OPENAI_CLIP_URL, CLIP_WEIGHT_PATH, OPENAI_CLIP_SHA256)\n',
    "print('Torch:', torch.__version__)\n",
    "print('Requested training processes:', NUM_TRAINING_PROCESSES)\n",
    "print('CUDA is initialized only inside the standalone torchrun workers.')\n",
    '\n',
    '@dataclass(frozen=True)\n',
    'class Sample:\n',
    '    image_path: Path\n',
    '    mask_path: Optional[Path]\n',
    '    label: int\n',
    '    category: str\n',
    '\n',
    'def image_files(folder):\n',
    "    return sorted(path for path in folder.rglob('*') if path.is_file() and path.suffix in IMAGE_EXTENSIONS)\n",
    '\n',
    'def mvtec_test_samples(root):\n',
    '    root = Path(root)\n',
    '    samples = []\n',
    "    for category_dir in sorted(path for path in root.iterdir() if (path / 'test').is_dir()):\n",
    "        for defect_dir in sorted(path for path in (category_dir / 'test').iterdir() if path.is_dir()):\n",
    "            anomalous = defect_dir.name.lower() != 'good'\n",
    '            for image_path in image_files(defect_dir):\n',
    '                mask_path = None\n',
    '                if anomalous:\n',
    '                    relative = image_path.relative_to(defect_dir)\n',
    '                    candidates = [\n',
    "                        category_dir / 'ground_truth' / defect_dir.name / relative.with_name(relative.stem + '_mask.png'),\n",
    "                        category_dir / 'ground_truth' / defect_dir.name / relative.with_suffix('.png'),\n",
    '                    ]\n',
    '                    mask_path = next((path for path in candidates if path.exists()), None)\n',
    '                    if mask_path is None:\n',
    "                        raise FileNotFoundError(f'Missing MVTec anomaly mask for {image_path}')\n",
    '                samples.append(Sample(image_path, mask_path, int(anomalous), category_dir.name))\n',
    '    if not samples:\n',
    "        raise RuntimeError(f'No MVTec test samples found under {root}')\n",
    '    return samples\n',
    '\n',
    'def resolve_visa_path(root, value):\n',
    "    value = str(value or '').strip()\n",
    '    if not value:\n',
    '        return None\n',
    '    raw = Path(value)\n',
    '    candidates = [raw] if raw.is_absolute() else [root / raw]\n',
    '    for candidate in candidates:\n',
    '        if candidate.exists():\n',
    '            return candidate\n',
    '    return None\n',
    '\n',
    'def visa_test_samples(root):\n',
    '    root = Path(root)\n',
    "    split_file = root / 'split_csv' / '1cls.csv'\n",
    '    if not split_file.exists():\n',
    "        raise FileNotFoundError(f'Official VisA split not found: {split_file}')\n",
    '    samples = []\n',
    "    with split_file.open('r', encoding='utf-8-sig', newline='') as handle:\n",
    '        for row in csv.DictReader(handle):\n',
    "            if str(row.get('split', '')).strip().lower() != 'test':\n",
    '                continue\n',
    "            category = str(row.get('object', '')).strip()\n",
    "            image_path = resolve_visa_path(root, row.get('image'))\n",
    '            if image_path is None:\n',
    '                raise FileNotFoundError(f"VisA image listed in split CSV is missing: {row.get(\'image\')}")\n',
    "            label_text = str(row.get('label', '')).strip().lower()\n",
    "            anomalous = label_text not in {'normal', 'good', '0', 'false'}\n",
    "            mask_path = resolve_visa_path(root, row.get('mask')) if anomalous else None\n",
    '            if anomalous and mask_path is None:\n',
    '                raise FileNotFoundError(f"VisA anomaly mask is missing: {row.get(\'mask\')}")\n',
    '            samples.append(Sample(image_path, mask_path, int(anomalous), category))\n',
    '    if not samples:\n',
    "        raise RuntimeError(f'No VisA test samples found through {split_file}')\n",
    '    return samples\n',
    '\n',
    'class FECLIPTrainingDataset(Dataset):\n',
    '    def __init__(self, dataset_name, root, image_size=336):\n',
    '        self.dataset_name = dataset_name\n',
    '        self.root = Path(root)\n',
    '        self.image_size = image_size\n',
    '        if not self.root.exists():\n',
    "            raise FileNotFoundError(f'Dataset root does not exist: {self.root}')\n",
    "        self.samples = mvtec_test_samples(self.root) if dataset_name == 'MVTec' else visa_test_samples(self.root)\n",
    '\n',
    '    def __len__(self):\n',
    '        return len(self.samples)\n',
    '\n',
    '    def __getitem__(self, index):\n',
    '        sample = self.samples[index]\n',
    "        image = Image.open(sample.image_path).convert('RGB')\n",
    '        image = TF.resize(image, [self.image_size, self.image_size], interpolation=InterpolationMode.BICUBIC, antialias=True)\n',
    '        image = TF.to_tensor(image)\n',
    '        image = TF.normalize(image, CLIP_MEAN, CLIP_STD)\n',
    '        if sample.mask_path is None:\n',
    '            mask = torch.zeros(1, self.image_size, self.image_size, dtype=torch.float32)\n',
    '        else:\n',
    "            mask_image = Image.open(sample.mask_path).convert('L')\n",
    '            mask_image = TF.resize(mask_image, [self.image_size, self.image_size], interpolation=InterpolationMode.NEAREST)\n',
    '            mask = (TF.pil_to_tensor(mask_image).float() > 0).float()\n',
    "        return {'image': image, 'mask': mask, 'label': torch.tensor(sample.label), 'category': sample.category}\n",
    '\n',
    'def orthonormal_dct_matrix(size, dtype=torch.float32):\n',
    '    matrix = torch.empty(size, size, dtype=dtype)\n',
    '    scale0 = math.sqrt(1.0 / size)\n',
    '    scale = math.sqrt(2.0 / size)\n',
    '    for frequency in range(size):\n',
    '        alpha = scale0 if frequency == 0 else scale\n',
    '        for position in range(size):\n',
    '            matrix[frequency, position] = alpha * math.cos(math.pi * (2 * position + 1) * frequency / (2 * size))\n',
    '    return matrix\n',
    '\n',
    'class FFEAdapter(nn.Module):\n',
    '    def __init__(self, channels, window_size=3):\n',
    '        super().__init__()\n',
    '        self.window_size = window_size\n',
    "        self.register_buffer('dct', orthonormal_dct_matrix(window_size), persistent=True)\n",
    '        self.linear = nn.Linear(channels, channels)\n',
    '        self.activation = nn.GELU()\n',
    '\n',
    '    def forward(self, features):\n',
    '        batch, channels, height, width = features.shape\n',
    '        p = self.window_size\n',
    '        if height % p or width % p:\n',
    "            raise ValueError(f'FFE requires a patch grid divisible by P={p}, got {height}x{width}')\n",
    '        blocks = features.unfold(2, p, p).unfold(3, p, p)  # B,C,H/P,W/P,P,P\n',
    '        dct = self.dct.to(device=features.device, dtype=features.dtype)\n',
    "        frequency = torch.einsum('ip,bcxypq,jq->bcxyij', dct, blocks, dct)\n",
    '        frequency = frequency.permute(0, 2, 3, 4, 5, 1)\n',
    '        frequency = self.activation(self.linear(frequency))\n',
    '        frequency = frequency.permute(0, 5, 1, 2, 3, 4)\n',
    "        spatial = torch.einsum('ip,bcxyij,jq->bcxypq', dct, frequency, dct)\n",
    '        return spatial.permute(0, 1, 2, 4, 3, 5).reshape(batch, channels, height, width)\n',
    '\n',
    'class LFSAdapter(nn.Module):\n',
    '    def __init__(self, channels, window_size=3, conv_kernel=1):\n',
    '        super().__init__()\n',
    '        if window_size % 2 != 1:\n',
    "            raise ValueError('Paper-faithful same-layout sliding DCT requires an odd Q')\n",
    '        self.window_size = window_size\n',
    "        self.register_buffer('dct', orthonormal_dct_matrix(window_size), persistent=True)\n",
    '        self.conv = nn.Conv2d(channels, channels, conv_kernel, padding=conv_kernel // 2)\n',
    '        self.activation = nn.GELU()\n',
    '\n',
    '    def forward(self, features):\n',
    '        batch, channels, height, width = features.shape\n',
    '        q = self.window_size\n',
    '        patches = F.unfold(features, kernel_size=q, padding=q // 2, stride=1)\n',
    '        patches = patches.view(batch, channels, q, q, height, width)\n',
    '        dct = self.dct.to(device=features.device, dtype=features.dtype)\n',
    "        frequency = torch.einsum('ip,bcpqhw,jq->bcijhw', dct, patches, dct)\n",
    '        mean_frequency = frequency.mean(dim=(2, 3))  # mean all QxQ response groups\n',
    '        return self.activation(self.conv(mean_frequency))\n',
    '\n',
    'class FocalLoss(nn.Module):\n',
    '    # Matches the AnomalyCLIP loss used by the training protocol FE-CLIP follows.\n',
    '    def __init__(self, gamma=2.0, smooth=1e-5):\n',
    '        super().__init__()\n',
    '        self.gamma = gamma\n',
    '        self.smooth = smooth\n',
    '\n',
    '    def forward(self, probabilities, target):\n',
    '        classes = probabilities.shape[1]\n',
    '        flattened = probabilities.permute(0, 2, 3, 1).reshape(-1, classes)\n',
    '        target = target.squeeze(1).reshape(-1).long()\n',
    '        one_hot = F.one_hot(target, num_classes=classes).to(flattened.dtype)\n',
    '        one_hot = one_hot.clamp(self.smooth / (classes - 1), 1.0 - self.smooth)\n',
    '        pt = (one_hot * flattened).sum(dim=1) + self.smooth\n',
    '        return (-(1.0 - pt).pow(self.gamma) * pt.log()).mean()\n',
    '\n',
    'class BinaryDiceLoss(nn.Module):\n',
    '    def __init__(self, smooth=1.0):\n',
    '        super().__init__()\n',
    '        self.smooth = smooth\n',
    '\n',
    '    def forward(self, probability, target):\n',
    '        probability = probability.reshape(probability.shape[0], -1)\n',
    '        target = target.reshape(target.shape[0], -1)\n',
    '        intersection = (probability * target).sum(dim=1)\n',
    '        dice = (2.0 * intersection + self.smooth) / (probability.sum(dim=1) + target.sum(dim=1) + self.smooth)\n',
    '        return 1.0 - dice.mean()\n',
    '\n',
    'class FECLIP(nn.Module):\n',
    '    def __init__(self, clip_model, stage_endpoints=(6, 12, 18, 24), p=3, q=3, frequency_lambda=0.1, lfs_kernel=1):\n',
    '        super().__init__()\n',
    '        self.clip = clip_model\n',
    '        self.stage_endpoints = tuple(stage_endpoints)\n',
    '        self.frequency_lambda = frequency_lambda\n',
    '        visual = self.clip.visual\n',
    '        self.width = visual.conv1.out_channels\n',
    '        self.embed_dim = visual.proj.shape[1]\n',
    '        self.grid_size = visual.input_resolution // visual.conv1.kernel_size[0]\n',
    '        self.ffe_adapters = nn.ModuleList([FFEAdapter(self.width, p) for _ in self.stage_endpoints])\n',
    '        self.lfs_adapters = nn.ModuleList([LFSAdapter(self.width, q, lfs_kernel) for _ in self.stage_endpoints])\n',
    '        # Section 3.2 calls this a single learnable fc shared across stage maps.\n',
    '        self.patch_projection = nn.Linear(self.width, self.embed_dim, bias=True)\n',
    '        for parameter in self.clip.parameters():\n',
    '            parameter.requires_grad_(False)\n',
    '        tokenized = clip.tokenize([NORMAL_PROMPT, ABNORMAL_PROMPT])\n',
    '        with torch.no_grad():\n',
    '            text = self.clip.encode_text(tokenized.to(next(self.clip.parameters()).device)).float()\n',
    '            text = F.normalize(text, dim=-1)\n',
    "        self.register_buffer('text_features', text, persistent=True)\n",
    '\n',
    '    def trainable_parameters(self):\n',
    '        modules = [self.ffe_adapters, self.lfs_adapters, self.patch_projection]\n',
    '        return [parameter for module in modules for parameter in module.parameters()]\n',
    '\n',
    '    def train(self, mode=True):\n',
    '        super().train(mode)\n',
    '        self.clip.eval()  # CLIP stays frozen/eval exactly as stated by the paper\n',
    '        return self\n',
    '\n',
    '    def _probabilities(self, visual_features):\n',
    '        visual_features = F.normalize(visual_features.float(), dim=-1)\n',
    '        scale = self.clip.logit_scale.exp().detach().float()\n',
    '        return (scale * visual_features @ self.text_features.float().T).softmax(dim=-1)\n',
    '\n',
    '    def forward(self, images):\n',
    '        visual = self.clip.visual\n',
    '        x = visual.conv1(images.to(dtype=visual.conv1.weight.dtype))\n',
    '        batch, channels, grid_h, grid_w = x.shape\n',
    '        if (grid_h, grid_w) != (self.grid_size, self.grid_size):\n',
    "            raise ValueError(f'Expected {self.grid_size}x{self.grid_size} CLIP patch grid, got {grid_h}x{grid_w}')\n",
    '        x = x.reshape(batch, channels, grid_h * grid_w).permute(0, 2, 1)\n',
    '        class_token = visual.class_embedding.to(x.dtype) + torch.zeros(batch, 1, channels, device=x.device, dtype=x.dtype)\n',
    '        x = torch.cat([class_token, x], dim=1)\n',
    '        x = visual.ln_pre(x + visual.positional_embedding.to(x.dtype))\n',
    '        x = x.permute(1, 0, 2)  # L,B,C for OpenAI CLIP residual blocks\n',
    '\n',
    '        class_probabilities = []\n',
    '        segmentation_probabilities = []\n',
    '        block_start = 0\n',
    '        for stage_index, block_end in enumerate(self.stage_endpoints):\n',
    '            for block in visual.transformer.resblocks[block_start:block_end]:\n',
    '                x = block(x)\n',
    '            block_start = block_end\n',
    '            cls = x[0]\n',
    '            patches = x[1:].permute(1, 2, 0).reshape(batch, channels, grid_h, grid_w)\n',
    '            frequency = self.ffe_adapters[stage_index](patches) + self.lfs_adapters[stage_index](patches)\n',
    '            enhanced = self.frequency_lambda * frequency + (1.0 - self.frequency_lambda) * patches\n',
    '\n',
    '            # Frozen standard CLIP class-token projection head at every stage.\n',
    '            projected_cls = visual.ln_post(cls) @ visual.proj\n',
    '            class_probabilities.append(self._probabilities(projected_cls))\n',
    '            projected_patches = self.patch_projection(enhanced.permute(0, 2, 3, 1))\n',
    '            segmentation_probabilities.append(self._probabilities(projected_patches))\n',
    '\n',
    '            enhanced_tokens = enhanced.flatten(2).permute(2, 0, 1)\n',
    '            x = torch.cat([cls.unsqueeze(0), enhanced_tokens], dim=0)\n',
    '\n',
    '        return class_probabilities, segmentation_probabilities\n',
    '\n',
    'def build_feclip(device):\n',
    "    backbone, _ = clip.load(str(CLIP_WEIGHT_PATH), device='cpu', jit=False)\n",
    '    backbone = backbone.float().to(device).eval()\n',
    '    model = FECLIP(\n',
    '        backbone, stage_endpoints=STAGE_ENDPOINTS, p=FFE_WINDOW, q=LFS_WINDOW,\n',
    '        frequency_lambda=FREQUENCY_LAMBDA, lfs_kernel=LFS_CONV_KERNEL,\n',
    '    ).to(device)\n',
    '    model.train()\n',
    '    return model\n',
    '\n',
    "dataset_root = MVTEC_PATH if TRAIN_DATASET == 'MVTec' else VISA_PATH\n",
    'train_dataset = FECLIPTrainingDataset(TRAIN_DATASET, dataset_root, IMAGE_SIZE)\n',
    'labels = np.array([sample.label for sample in train_dataset.samples])\n',
    'categories = sorted({sample.category for sample in train_dataset.samples})\n',
    "print('Samples:', len(train_dataset))\n",
    "print('Normal/anomalous:', int((labels == 0).sum()), int((labels == 1).sum()))\n",
    "print('Categories:', categories)\n",
    'optimizer_steps_per_epoch = math.ceil(len(train_dataset) / TOTAL_BATCH_SIZE)\n',
    "print('Approximate optimizer steps per epoch:', optimizer_steps_per_epoch)\n",
    "print('Approximate optimizer steps for all 9 epochs:', optimizer_steps_per_epoch * EPOCHS)\n",
    'first = train_dataset[0]\n',
    "assert first['image'].shape == (3, IMAGE_SIZE, IMAGE_SIZE)\n",
    "assert first['mask'].shape == (1, IMAGE_SIZE, IMAGE_SIZE)\n",
    '\n',
    'focal_loss = FocalLoss(FOCAL_GAMMA, FOCAL_SMOOTH)\n',
    'dice_loss = BinaryDiceLoss(DICE_SMOOTH)\n',
    '\n',
    'def feclip_loss(class_probabilities, segmentation_probabilities, labels, masks):\n',
    '    labels_float = labels.float()\n',
    '    class_losses = [F.binary_cross_entropy(probs[:, 1].clamp(1e-6, 1 - 1e-6), labels_float) for probs in class_probabilities]\n',
    '    mask_losses = []\n',
    '    for probs in segmentation_probabilities:\n',
    '        probs = probs.permute(0, 3, 1, 2)\n',
    "        probs = F.interpolate(probs, size=masks.shape[-2:], mode='bilinear', align_corners=False)\n",
    '        mask_losses.append(focal_loss(probs, masks) + dice_loss(probs[:, 1:2], masks))\n',
    '    loss_cls = torch.stack(class_losses).mean()\n',
    '    loss_mask = torch.stack(mask_losses).mean()\n',
    '    return loss_cls + loss_mask, loss_cls.detach(), loss_mask.detach()\n',
    '\n',
    'def checkpoint_config(seed):\n',
    '    return {\n',
    "        'paper': 'Gong et al., FE-CLIP, ICCV 2025',\n",
    "        'implementation': 'independent_paper_reconstruction',\n",
    "        'train_dataset': TRAIN_DATASET,\n",
    "        'intended_target': 'MVTec' if TRAIN_DATASET == 'VisA' else 'VisA_and_other_datasets',\n",
    "        'backbone': 'OpenAI CLIP ViT-L/14@336px',\n",
    "        'clip_sha256': OPENAI_CLIP_SHA256,\n",
    "        'image_size': IMAGE_SIZE, 'epochs': EPOCHS, 'learning_rate': LEARNING_RATE,\n",
    "        'total_batch_size': TOTAL_BATCH_SIZE, 'gpu_micro_batch_size': GPU_MICRO_BATCH_SIZE,\n",
    "        'optimizer': 'Adam', 'adam_betas': ADAM_BETAS, 'adam_eps': ADAM_EPS,\n",
    "        'stage_endpoints': STAGE_ENDPOINTS, 'ffe_window_p': FFE_WINDOW, 'lfs_window_q': LFS_WINDOW,\n",
    "        'frequency_lambda': FREQUENCY_LAMBDA, 'dct_norm': DCT_NORM, 'lfs_conv_kernel': LFS_CONV_KERNEL,\n",
    "        'normal_prompt': NORMAL_PROMPT, 'abnormal_prompt': ABNORMAL_PROMPT,\n",
    "        'focal_gamma': FOCAL_GAMMA, 'focal_smooth': FOCAL_SMOOTH, 'dice_smooth': DICE_SMOOTH,\n",
    "        'use_amp': USE_AMP, 'seed': seed, 'torch_version': torch.__version__,\n",
    "        'paper_underspecified_choices': [\n",
    "            'stage_endpoints', 'dct_norm', 'lfs_conv_kernel', 'optimizer_betas_and_eps',\n",
    "            'focal_parameters', 'resize_rule', 'seed', 'parameter_initialization',\n",
    "            'intermediate_class_projection_layernorm', 'learnable_patch_fc_optimization',\n",
    '        ],\n',
    '    }\n',
    '\n',
    'def train_one_seed(seed, accelerator, gradient_accumulation_steps):\n',
    '    set_seed(seed)\n',
    '    generator = torch.Generator().manual_seed(seed)\n',
    '    loader = DataLoader(\n',
    '        train_dataset, batch_size=GPU_MICRO_BATCH_SIZE, shuffle=True, generator=generator,\n',
    '        num_workers=NUM_WORKERS, pin_memory=True, drop_last=False,\n',
    '    )\n',
    '    model = build_feclip(accelerator.device)\n',
    '    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)\n',
    '    frozen = sum(parameter.numel() for parameter in model.parameters() if not parameter.requires_grad)\n',
    '    assert all(not parameter.requires_grad for parameter in model.clip.parameters())\n',
    '    assert len(model.ffe_adapters) == len(model.lfs_adapters) == 4\n',
    '    optimizer = torch.optim.Adam(\n',
    '        model.trainable_parameters(), lr=LEARNING_RATE, betas=ADAM_BETAS, eps=ADAM_EPS, weight_decay=0.0,\n',
    '    )\n',
    '    model, optimizer, loader = accelerator.prepare(model, optimizer, loader)\n',
    "    run_dir = OUTPUT_ROOT / f'train_on_{TRAIN_DATASET.lower()}_seed_{seed}'\n",
    '    if accelerator.is_main_process:\n',
    '        run_dir.mkdir(parents=True, exist_ok=True)\n',
    "        print(f'DDP workers: {accelerator.num_processes}; accumulation: {gradient_accumulation_steps}')\n",
    "        print(f'Trainable parameters: {trainable:,}; frozen CLIP parameters: {frozen:,}')\n",
    '    accelerator.wait_for_everyone()\n',
    '    history = []\n',
    '\n',
    '    for epoch in range(1, EPOCHS + 1):\n',
    '        epoch_started = time.perf_counter()\n',
    '        model.train()\n',
    "        if hasattr(loader, 'set_epoch'):\n",
    '            loader.set_epoch(epoch)\n',
    '        local_totals = torch.zeros(4, device=accelerator.device, dtype=torch.float64)\n',
    '        progress = tqdm(\n',
    "            loader, desc=f'seed {seed} epoch {epoch}/{EPOCHS}',\n",
    '            disable=not accelerator.is_local_main_process,\n',
    '        )\n',
    '        optimizer.zero_grad(set_to_none=True)\n',
    '        for batch in progress:\n',
    '            with accelerator.accumulate(model):\n',
    '                with accelerator.autocast():\n',
    "                    class_probs, segmentation_probs = model(batch['image'])\n",
    '                    loss, loss_cls, loss_mask = feclip_loss(\n',
    "                        class_probs, segmentation_probs, batch['label'], batch['mask'],\n",
    '                    )\n',
    '                accelerator.backward(loss)\n',
    '                optimizer.step()\n',
    '                optimizer.zero_grad(set_to_none=True)\n',
    "            count = batch['image'].shape[0]\n",
    '            local_totals += torch.stack([\n',
    '                loss.detach() * count, loss_cls * count, loss_mask * count,\n',
    '                torch.tensor(count, device=accelerator.device, dtype=loss.dtype),\n',
    '            ]).to(torch.float64)\n',
    '            if accelerator.sync_gradients:\n',
    '                progress.set_postfix(loss=f"{loss.detach().item():.4f}")\n',
    '\n',
    "        totals = accelerator.reduce(local_totals, reduction='sum')\n",
    '        if accelerator.is_main_process:\n',
    '            sample_count = int(totals[3].item())\n',
    '            record = {\n',
    "                'loss': totals[0].item() / sample_count,\n",
    "                'cls': totals[1].item() / sample_count,\n",
    "                'mask': totals[2].item() / sample_count,\n",
    "                'samples': sample_count, 'epoch': epoch,\n",
    '            }\n',
    "            record['epoch_minutes'] = (time.perf_counter() - epoch_started) / 60.0\n",
    "            record['estimated_remaining_hours'] = record['epoch_minutes'] * (EPOCHS - epoch) / 60.0\n",
    '            history.append(record)\n',
    '            core_model = accelerator.unwrap_model(model)\n',
    '            config = checkpoint_config(seed)\n',
    '            config.update({\n',
    "                'distributed_backend': 'Accelerate_DDP',\n",
    "                'num_training_processes': accelerator.num_processes,\n",
    "                'gradient_accumulation_steps': gradient_accumulation_steps,\n",
    '            })\n',
    '            checkpoint = {\n',
    "                'epoch': epoch, 'config': config, 'history': history,\n",
    "                'ffe_adapters': core_model.ffe_adapters.state_dict(),\n",
    "                'lfs_adapters': core_model.lfs_adapters.state_dict(),\n",
    "                'patch_projection': core_model.patch_projection.state_dict(),\n",
    "                'optimizer': optimizer.state_dict(),\n",
    '            }\n',
    "            checkpoint_path = run_dir / f'feclip_train_on_{TRAIN_DATASET.lower()}_epoch_{epoch:02d}.pth'\n",
    '            torch.save(checkpoint, checkpoint_path)\n',
    "            (run_dir / 'history.json').write_text(json.dumps(history, indent=2), encoding='utf-8')\n",
    '            print(record)\n',
    '        accelerator.wait_for_everyone()\n',
    '\n',
    '    del model, optimizer, loader\n',
    '    accelerator.free_memory()\n',
    '\n',
    'def distributed_training_entrypoint():\n',
    "    mixed_precision = 'fp16' if USE_AMP else 'no'\n",
    '    accumulation = TOTAL_BATCH_SIZE // (NUM_TRAINING_PROCESSES * GPU_MICRO_BATCH_SIZE)\n',
    '    accelerator = Accelerator(gradient_accumulation_steps=accumulation, mixed_precision=mixed_precision)\n',
    '    if accelerator.num_processes != NUM_TRAINING_PROCESSES:\n',
    '        raise RuntimeError(\n',
    "            f'Requested {NUM_TRAINING_PROCESSES} processes but Accelerate started {accelerator.num_processes}. '\n",
    "            'Select Kaggle T4 x2 or set NUM_TRAINING_PROCESSES=1.'\n",
    '        )\n',
    '    for seed in SEEDS:\n',
    '        train_one_seed(seed, accelerator, accumulation)\n',
    '    accelerator.end_training()\n',
    '\n',
    "def load_feclip_checkpoint(checkpoint_path, device=torch.device('cpu')):\n",
    "    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)\n",
    '    model = build_feclip(device)\n',
    "    model.ffe_adapters.load_state_dict(checkpoint['ffe_adapters'])\n",
    "    model.lfs_adapters.load_state_dict(checkpoint['lfs_adapters'])\n",
    "    model.patch_projection.load_state_dict(checkpoint['patch_projection'])\n",
    '    return model.eval(), checkpoint\n',
    '\n',
    "if __name__ == '__main__':\n",
    '    distributed_training_entrypoint()\n',
])
TRAINING_SCRIPT_SHA256 = '7911186759c93d44a3ba0506ebfd8b21bdf382e55e93829fe606f9720c73ff2a'
assert hashlib.sha256(training_script_source.encode('utf-8')).hexdigest() == TRAINING_SCRIPT_SHA256
runtime_config = {
    'WORKING_DIR': str(WORKING_DIR), 'MVTEC_PATH': str(MVTEC_PATH), 'VISA_PATH': str(VISA_PATH),
    'OUTPUT_ROOT': str(OUTPUT_ROOT), 'TRAIN_DATASET': TRAIN_DATASET,
    'IMAGE_SIZE': IMAGE_SIZE, 'EPOCHS': EPOCHS, 'LEARNING_RATE': LEARNING_RATE,
    'TOTAL_BATCH_SIZE': TOTAL_BATCH_SIZE, 'FFE_WINDOW': FFE_WINDOW, 'LFS_WINDOW': LFS_WINDOW,
    'FREQUENCY_LAMBDA': FREQUENCY_LAMBDA, 'NUM_STAGES': NUM_STAGES,
    'NORMAL_PROMPT': NORMAL_PROMPT, 'ABNORMAL_PROMPT': ABNORMAL_PROMPT,
    'NUM_TRAINING_PROCESSES': NUM_TRAINING_PROCESSES,
    'GPU_MICRO_BATCH_SIZE': GPU_MICRO_BATCH_SIZE, 'NUM_WORKERS': NUM_WORKERS, 'USE_AMP': USE_AMP,
    'SEEDS': list(SEEDS), 'STAGE_ENDPOINTS': list(STAGE_ENDPOINTS), 'DCT_NORM': DCT_NORM,
    'LFS_CONV_KERNEL': LFS_CONV_KERNEL, 'FOCAL_GAMMA': FOCAL_GAMMA,
    'FOCAL_SMOOTH': FOCAL_SMOOTH, 'DICE_SMOOTH': DICE_SMOOTH,
    'ADAM_BETAS': list(ADAM_BETAS), 'ADAM_EPS': ADAM_EPS,
    'OPENAI_CLIP_URL': OPENAI_CLIP_URL, 'OPENAI_CLIP_SHA256': OPENAI_CLIP_SHA256,
    'CLIP_COMMIT': CLIP_COMMIT, 'CLIP_WEIGHT_PATH': str(CLIP_WEIGHT_PATH),
}
worker_path = WORKING_DIR / 'feclip_torchrun_worker.py'
worker_path.write_text(training_script_source, encoding='utf-8')
launch_environment = os.environ.copy()
for distributed_variable in ('RANK', 'WORLD_SIZE', 'LOCAL_RANK', 'LOCAL_WORLD_SIZE', 'MASTER_ADDR', 'MASTER_PORT'):
    launch_environment.pop(distributed_variable, None)
launch_environment['FECLIP_RUNTIME_CONFIG_JSON'] = json.dumps(runtime_config)
launch_environment['PYTHONUNBUFFERED'] = '1'
launch_command = [
    sys.executable, '-u', '-m', 'torch.distributed.run', '--standalone',
    f'--nproc_per_node={NUM_TRAINING_PROCESSES}', str(worker_path),
]
print('Runtime training source passed to workers:', runtime_config['TRAIN_DATASET'])
print(f'Launching training in {NUM_TRAINING_PROCESSES} fresh CUDA worker(s):', ' '.join(launch_command))
subprocess.run(launch_command, env=launch_environment, check=True)
completed_runs = [OUTPUT_ROOT / f'train_on_{TRAIN_DATASET.lower()}_seed_{seed}' for seed in SEEDS]
print('Completed:', completed_runs)


In [ ]:
# Validate the final checkpoint structure and create one downloadable archive.
for run_dir in completed_runs:
    final_checkpoint = run_dir / f'feclip_train_on_{TRAIN_DATASET.lower()}_epoch_{EPOCHS:02d}.pth'
    restored_model, payload = load_feclip_checkpoint(final_checkpoint, torch.device('cpu'))
    assert payload['epoch'] == EPOCHS
    assert payload['config']['train_dataset'] == TRAIN_DATASET
    del restored_model
    print('Validated:', final_checkpoint, f'{final_checkpoint.stat().st_size / 2**20:.2f} MiB')

archive_base = WORKING_DIR / f'feclip_train_on_{TRAIN_DATASET.lower()}_checkpoints'
archive_path = shutil.make_archive(str(archive_base), 'zip', OUTPUT_ROOT)
print('Downloadable archive:', archive_path)